## Análise de Dados de Corridas de Táxi

Este projeto tem como objetivo analisar dados de corridas de táxi, aplicando etapas de validação, limpeza e transformação para identificar padrões de comportamento dos usuários.

Os dados foram processados utilizando DuckDB e armazenados em formato parquet, seguindo uma abordagem de pipeline de dados.

In [5]:
import duckdb

# conexão com DuckDB para executar consultas no parquet
con = duckdb.connect()

In [6]:
import os

# verifica se o arquivo de amostra existe
os.path.exists("../data/processed/yellow_tripdata_2026-01_sample.parquet")

True

In [7]:
# leitura leve do dataset tratado (apenas algumas colunas)
# evita travamentos no notebook
df_sample = con.execute("""
    SELECT
        trip_distance,
        total_amount,
        trip_duration_min,
        pickup_hour
    FROM read_parquet('../data/processed/yellow_tripdata_2026-01_sample.parquet')
    LIMIT 5
""").df()

df_sample

,trip_distance,total_amount,trip_duration_min,pickup_hour
0,5.30,36.16,20,19
1,2.11,18.54,8,8
2,6.64,48.21,24,20
3,1.38,22.38,10,19
4,4.35,32.96,25,17


In [8]:
# contagem total de registros na amostra
volume_sample = con.execute("""
SELECT COUNT(*) AS total_linhas
FROM read_parquet('../data/processed/yellow_tripdata_2026-01_sample.parquet')
""").df()

volume_sample

,total_linhas
0,95681


### Validação do Volume de Dados

A amostra utilizada contém aproximadamente 95 mil registros, sendo suficiente para análise exploratória sem comprometer a performance.

Essa abordagem permite trabalhar com grandes volumes de dados de forma eficiente.

In [9]:
# cálculo de métricas médias do dataset
# ajuda a entender o comportamento geral das corridas
metricas = con.execute("""
SELECT
    ROUND(AVG(trip_distance), 2) AS distancia_media,
    ROUND(AVG(total_amount), 2) AS valor_medio,
    ROUND(AVG(trip_duration_min), 2) AS duracao_media
FROM read_parquet('../data/processed/yellow_tripdata_2026-01_sample.parquet')
""").df()

metricas

,distancia_media,valor_medio,duracao_media
0,9.48,29.69,17.35


### Análise de Métricas Gerais

- Distância média: ~9.5 km  
- Duração média: ~17 minutos  
- Valor médio: ~$30  

**Interpretação:**

Os valores indicam corridas de curta a média distância, com duração e custo consistentes, sugerindo que o processo de limpeza removeu corretamente registros inválidos.

In [10]:
# distribuição das corridas ao longo do dia
# permite identificar horários de maior demanda
corridas_hora = con.execute("""
SELECT
    pickup_hour,
    COUNT(*) AS total_corridas
FROM read_parquet('../data/processed/yellow_tripdata_2026-01_sample.parquet')
GROUP BY pickup_hour
ORDER BY pickup_hour
""").df()

corridas_hora

,pickup_hour,total_corridas
0,0,2950
1,1,2057
2,2,1420
3,3,986
4,4,703
5,5,852
6,6,1563
7,7,2963
8,8,4018
9,9,4108


### Distribuição de Corridas por Hora

Observa-se:

- Baixa demanda durante a madrugada (0h–5h)
- Crescimento a partir das 6h
- Pico entre 11h e 14h

**Insight:**

O padrão indica comportamento típico de mobilidade urbana, com maior utilização do serviço em horários comerciais e de deslocamento.

In [1]:
import duckdb

# cria conexão
con = duckdb.connect()

# instala e carrega extensão para acessar S3
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# extensão para integração com credenciais AWS
con.execute("INSTALL aws;")
con.execute("LOAD aws;")

# usa as credenciais do aws configure
con.execute("""
    CREATE OR REPLACE SECRET (
        TYPE s3,
        PROVIDER credential_chain,
        REGION 'sa-east-1'
    );
""")

# leitura direta do arquivo no S3
df_s3 = con.execute("""
    SELECT
        VendorID,
        tpep_pickup_datetime,
        trip_distance,
        total_amount
    FROM read_parquet('s3://taxi-data-aline-2026/yellow_tripdata_2026-01.parquet')
    LIMIT 5
""").df()

df_s3

,VendorID,tpep_pickup_datetime,trip_distance,total_amount
0,2,2026-01-01 00:54:04,0.97,15.86
1,1,2026-01-01 00:34:04,0.90,13.65
2,1,2026-01-01 00:57:06,1.40,18.95
3,2,2026-01-01 00:15:22,5.58,55.56
4,2,2026-01-01 00:27:13,2.16,23.10
